# Region survey + sliding Lipschitz quotients on Dandiset 000363

Limited build, per the folder README: we only load the data and look, no modelling.

1. **Region survey** — stream the electrodes table of every session of
   [Dandiset 000363](https://dandiarchive.org/dandiset/000363) (Chen et al. 2024,
   brain-wide activity during memory-guided movement) and tabulate which brain
   regions each session records. Then: which sessions hold the most regions at
   once, and which region *combinations* are shared by the most sessions.
2. **One session** — load trials + good units, bin spikes at 10 ms, SVD the
   correlation matrix of the binned counts, keep the top K PCs (elbow, or 10,
   whichever comes first), and compute the local Lipschitz constant of the
   projected trajectory over time (`||dx||/dt` and `||dv||/dt`, as in the
   bouncing-ball demo) against the trial structure. Then the same trace per
   brain region: jumps shared across regions at the same moment are the
   guard-suggestive kind.

Nothing is downloaded: files are streamed from S3 with `remfile` byte ranges, so
even the ~300 GB raw-ephys sessions only cost the arrays we touch. The survey
caches to a CSV next to this notebook and resumes if interrupted.

In [ ]:
# ----------------------------- config ---------------------------------
import time
import itertools
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import remfile
import pynwb
from dandi.dandiapi import DandiAPIClient

DANDISET_ID = "000363"
SURVEY_CACHE = Path("region_survey_000363.csv")  # written next to this notebook
SURVEY_LIMIT = None      # int for a quick pass (e.g. 25); None = all 174 sessions

CHOSEN_PATH = None       # None = auto-pick the surveyed session with most regions

# binning / embedding
BIN_MS = 10.0            # spike-count bin width for the covariance/correlation
K_MAX = 10               # keep top-K PCs: the spectrum's elbow, capped at this
PAD_S = 5.0              # seconds kept before the first / after the last trial
MIN_RATE_HZ = 0.1        # drop near-silent units
MIN_UNITS_REGION = 15    # regions with fewer good units skip the per-region trace

# detection, same spirit as the bouncing-ball demo: free flight has a steady
# quotient, a reset is a huge isolated spike above a robust baseline
THRESH_MULT = 5.0        # threshold = THRESH_MULT * median(||dv||/dt)

ZOOM_START = None        # seconds; None = auto (a little before the 10th go cue)
ZOOM_DUR = 30.0          # seconds shown in the zoomed trace figure

## 1. Region survey across all sessions

The DANDI asset metadata does not carry region labels, so for each NWB file we
stream only what we need: `electrodes/location`, the unit count, and the
unit → electrode mapping (so regions are counted by *units actually recorded
there*, not just by electrode placement). A couple of small int/str arrays per
file — a few seconds each, not 300 GB.

In [ ]:
def _decode(arr):
    """HDF5 string data to a plain numpy array of str."""
    return np.array([x.decode() if isinstance(x, (bytes, bytearray)) else str(x)
                     for x in np.asarray(arr)])


def _ragged(group, col):
    """A DynamicTable ragged column as a list of per-row arrays."""
    data = group[col][:]
    if col + "_index" in group:
        return np.split(data, group[col + "_index"][:-1])
    return [np.atleast_1d(v) for v in data]


def open_stream(asset):
    url = asset.get_content_url(follow_redirects=1, strip_query=True)
    return h5py.File(remfile.File(url), "r")


def survey_session(asset):
    """One row of the survey: regions present and units per region."""
    sub, fname = asset.path.split("/", 1)
    row = {"path": asset.path, "subject": sub.replace("sub-", ""),
           "size_gb": round(asset.size / 1e9, 1),
           "n_electrodes": 0, "n_units": 0, "regions": "", "units_per_region": "",
           "error": ""}
    with open_stream(asset) as f:
        elec = f.get("general/extracellular_ephys/electrodes")
        if elec is None or "location" not in elec:
            row["error"] = "no electrodes table"
            return row
        loc = _decode(elec["location"][:])
        loc = np.array([s.strip() for s in loc])
        row["n_electrodes"] = len(loc)

        units = f.get("units")
        per_region = Counter()
        if units is not None and "id" in units:
            row["n_units"] = len(units["id"])
            if "electrodes" in units:
                # region of a unit = location of its first electrode
                first_elec = [int(e[0]) for e in _ragged(units, "electrodes") if len(e)]
                per_region = Counter(loc[first_elec])
        if not per_region:               # fall back to electrode placement
            per_region = Counter(loc)
        per_region = Counter({k: v for k, v in per_region.items()
                              if k and k.lower() not in ("none", "n/a", "unknown")})
        row["regions"] = ";".join(sorted(per_region))
        row["units_per_region"] = ";".join(
            "%s=%d" % kv for kv in sorted(per_region.items()))
    return row

In [ ]:
cache = {}
if SURVEY_CACHE.exists():
    cache = {r["path"]: dict(r) for _, r in pd.read_csv(
        SURVEY_CACHE, keep_default_na=False).iterrows()}
    print("cache: %d sessions already surveyed" % len(cache))

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(DANDISET_ID, "draft")
    assets = sorted((a for a in dandiset.get_assets() if a.path.endswith(".nwb")),
                    key=lambda a: a.path)
    if SURVEY_LIMIT is not None:
        assets = assets[:SURVEY_LIMIT]
    print("%d sessions to survey" % len(assets))

    for i, asset in enumerate(assets):
        if asset.path in cache and not cache[asset.path].get("error"):
            continue
        t0 = time.time()
        try:
            row = survey_session(asset)
        except Exception as exc:                       # keep going, log the file
            row = {"path": asset.path, "error": "%s: %s" % (type(exc).__name__, exc)}
        cache[asset.path] = row
        pd.DataFrame(cache.values()).to_csv(SURVEY_CACHE, index=False)
        print("[%3d/%3d] %5.1fs  %-60s %s" % (
            i + 1, len(assets), time.time() - t0, asset.path,
            row.get("regions", "") or row.get("error", "")), flush=True)

survey = pd.DataFrame(cache.values())
survey = survey[survey["error"].fillna("") == ""].reset_index(drop=True)
survey["n_regions"] = survey["regions"].str.split(";").map(
    lambda r: len([x for x in r if x]))
print("\nsurveyed OK: %d sessions, %d subjects" % (
    len(survey), survey["subject"].nunique()))

In [ ]:
# ---------------- sessions x regions presence matrix -------------------
region_sets = [set(r.split(";")) - {""} for r in survey["regions"]]
all_regions = sorted(set().union(*region_sets))
presence = np.array([[reg in s for reg in all_regions] for s in region_sets])
print("%d distinct region labels: %s" % (len(all_regions), ", ".join(all_regions)))

ranked = survey.sort_values(["n_regions", "n_units"], ascending=False)
print("\nsessions with the most regions at once:")
print(ranked[["path", "subject", "n_regions", "n_units", "regions"]]
      .head(12).to_string(index=False))

order = np.argsort(-survey["n_regions"].values, kind="stable")
fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
ax.imshow(presence[order].T, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_yticks(range(len(all_regions)))
ax.set_yticklabels(all_regions, fontsize=8)
ax.set_xlabel("session (sorted by number of regions)")
ax.set_title("which regions each session records")

ax = axes[1]
freq = presence.sum(0)
fo = np.argsort(freq)
ax.barh(range(len(all_regions)), freq[fo], color="tab:blue")
ax.set_yticks(range(len(all_regions)))
ax.set_yticklabels(np.array(all_regions)[fo], fontsize=8)
ax.set_xlabel("sessions containing region")
ax.set_title("region frequency")
plt.tight_layout()
plt.show()

In [ ]:
# -------- region combinations: what do many sessions share? ------------
# exact recorded sets
set_counts = Counter(frozenset(s) for s in region_sets)
print("most common exact region sets:")
for s, n in set_counts.most_common(10):
    print("  %3d sessions  (%d regions)  %s" % (n, len(s), ", ".join(sorted(s))))

# best k-region combination shared by the most sessions: for each k, the combo
# of regions that the largest number of sessions ALL contain
print("\nbest shared combination per combination size k:")
rows = []
for k in range(2, min(len(all_regions), 8) + 1):
    best = []
    for combo in itertools.combinations(range(len(all_regions)), k):
        n = int(presence[:, combo].all(axis=1).sum())
        if n:
            best.append((n, combo))
    if not best:
        break
    best.sort(reverse=True)
    for n, combo in best[:3]:
        rows.append({"k": k, "sessions": n,
                     "regions": ", ".join(all_regions[i] for i in combo)})
        print("  k=%d  %3d sessions  %s" % (k, n, rows[-1]["regions"]))
combo_table = pd.DataFrame(rows)

## 2. One session: trial structure vs the Lipschitz constant of the top PCs

Pick the surveyed session with the most simultaneous regions (override with
`CHOSEN_PATH`), stream it, and build a light units table — spike times, region,
quality — without pulling waveforms or raw ephys.

In [ ]:
if CHOSEN_PATH is None:
    CHOSEN_PATH = ranked.iloc[0]["path"]
print("loading:", CHOSEN_PATH)

with DandiAPIClient() as client:
    asset = client.get_dandiset(DANDISET_ID, "draft").get_asset_by_path(CHOSEN_PATH)
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)

h5f = h5py.File(remfile.File(s3_url), "r")
io = pynwb.NWBHDF5IO(file=h5f, load_namespaces=True)
nwb = io.read()
print(nwb.session_description)

# --- light units table straight from the HDF5 group (skips waveforms) ---
ug = h5f["units"]
print("units table columns:", [k for k in ug.keys() if not k.endswith("_index")])

spike_trains = _ragged(ug, "spike_times")
elec_loc = np.array([s.strip() for s in
                     _decode(h5f["general/extracellular_ephys/electrodes/location"][:])])
unit_region = np.array([elec_loc[int(e[0])] if len(e) else ""
                        for e in _ragged(ug, "electrodes")])

quality = None
for cand in ("classification", "unit_quality", "quality", "label"):
    if cand in ug:
        quality = np.array([s.strip() for s in _decode(ug[cand][:])])
        print("quality column: %r, values: %s" % (cand, dict(Counter(quality))))
        break

units = pd.DataFrame({"region": unit_region,
                      "n_spikes": [len(s) for s in spike_trains]})
units["spike_times"] = spike_trains
if quality is not None:
    units["quality"] = quality
    units = units[units["quality"].str.lower().str.contains("good")]

trials_df = nwb.trials.to_dataframe()
t0 = float(trials_df["start_time"].min()) - PAD_S
t1 = float(trials_df["stop_time"].max()) + PAD_S
units["rate_hz"] = units["n_spikes"] / (t1 - t0)
units = units[units["rate_hz"] >= MIN_RATE_HZ].reset_index(drop=True)
print("\n%d units kept, by region:" % len(units))
print(units["region"].value_counts().to_string())

In [ ]:
# ------------------------- trial structure -----------------------------
print("%d trials, columns:" % len(trials_df), list(trials_df.columns))
display(trials_df.head())

# task epoch events (sample / delay / go cues) live in BehavioralEvents
events = {}
beh = nwb.acquisition.get("BehavioralEvents")
if beh is not None:
    for name, ts in beh.time_series.items():
        if any(k in name.lower() for k in ("sample", "delay", "go")):
            events[name] = np.asarray(ts.timestamps[:])
for name, tt in sorted(events.items()):
    print("%-28s %6d events" % (name, len(tt)))

go_key = next((k for k in events if "go" in k.lower()), None)
go_times = events.get(go_key)
print("\nusing %r as the go cue" % go_key)

In [ ]:
# -------- bin at 10 ms, SVD the correlation matrix, elbow-pick K ---------
dt = BIN_MS / 1000.0
edges = np.arange(t0, t1, dt)
tb = edges[:-1] + dt / 2

X = np.empty((len(tb), len(units)), dtype=np.float32)
for j, st in enumerate(units["spike_times"]):
    X[:, j], _ = np.histogram(st, bins=edges)
print("binned counts:", X.shape, "(%.1f min at %g ms bins)" % (
    (t1 - t0) / 60, BIN_MS))


def corr_svd(counts):
    """Correlation matrix of binned counts -> (eigenvalues desc, eigenvectors,
    z-scored counts). Z-scoring the units makes cov(Z) the correlation of X."""
    Z = counts - counts.mean(0)
    Z /= np.maximum(counts.std(0), 1e-9)
    corr = (Z.T @ Z) / (len(Z) - 1)
    evals, evecs = np.linalg.eigh(corr)
    return evals[::-1], evecs[:, ::-1], Z


def elbow_k(evals, k_max, m=50):
    """Elbow = eigenvalue farthest below the chord through the leading
    spectrum; K = min(elbow, k_max) -- whichever comes first."""
    m = min(m, len(evals))
    y = evals[:m]
    chord = y[0] + (y[-1] - y[0]) * np.arange(m) / (m - 1)
    return min(int(np.argmax(chord - y)) + 1, k_max)


evals, evecs, Z = corr_svd(X)
K = elbow_k(evals, K_MAX)
x = Z @ evecs[:, :K]               # (time, K) trajectory
print("elbow at %d, K_MAX %d -> keeping K = %d PCs (%.1f%% of correlation "
      "variance)" % (elbow_k(evals, 10**9), K_MAX, K,
                     100 * evals[:K].sum() / evals.sum()))

fig, ax = plt.subplots(figsize=(6, 3))
n_show = min(30, len(evals))
ax.plot(np.arange(1, n_show + 1), evals[:n_show], "o-", color="tab:blue", ms=4)
ax.axvline(K, color="tab:red", ls="--", lw=1, label="K = %d" % K)
ax.set_xlabel("component")
ax.set_ylabel("eigenvalue of correlation matrix")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# -------------- sliding local Lipschitz quotients -----------------------
# As in the bouncing-ball demo: on a smooth flow ||dv||/dt stays near a steady
# baseline; a guard/reset shows up as an isolated spike orders of magnitude above.
Lp = np.linalg.norm(np.diff(x, axis=0), axis=1) / dt          # ||dp|| / dt
v = np.diff(x, axis=0) / dt
Lv = np.linalg.norm(np.diff(v, axis=0), axis=1) / dt          # ||dv|| / dt
t_p = tb[:-1] + dt / 2
t_v = tb[1:-1]

threshold = THRESH_MULT * np.median(Lv)
detected = Lv > threshold
det_times = t_v[detected]
print("threshold = %.3g;  %d detections in %.1f min (%.2f per min)" % (
    threshold, detected.sum(), (t1 - t0) / 60,
    detected.sum() / ((t1 - t0) / 60)))

# full-session overview
fig, ax = plt.subplots(figsize=(13, 3))
ax.semilogy(t_v, Lv, color="tab:purple", lw=0.3)
ax.axhline(threshold, color="k", ls=":", lw=1)
ax.plot(det_times, Lv[detected], ".", color="tab:red", ms=4)
ax.set_xlabel("time (s)")
ax.set_ylabel("||\u0394v|| / \u0394t")
ax.set_title("whole session: local Lipschitz quotient of the PCA trajectory")
plt.tight_layout()
plt.show()

In [ ]:
# -------------------- zoomed trace against the trials -------------------
if ZOOM_START is None:
    ZOOM_START = (go_times[9] - 5.0 if go_times is not None and len(go_times) > 10
                  else t0 + 60.0)
z0, z1 = ZOOM_START, ZOOM_START + ZOOM_DUR
mb = (tb >= z0) & (tb < z1)
mv = (t_v >= z0) & (t_v < z1)
mp = (t_p >= z0) & (t_p < z1)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                         gridspec_kw={"height_ratios": [1.2, 1]})
for ax in axes:
    for _, tr in trials_df.iterrows():
        if tr["stop_time"] > z0 and tr["start_time"] < z1:
            ax.axvspan(tr["start_time"], tr["stop_time"],
                       color="tab:blue", alpha=0.07, zorder=0)
    if go_times is not None:
        for g in go_times[(go_times >= z0) & (go_times < z1)]:
            ax.axvline(g, color="0.6", ls="--", lw=0.8, zorder=0)

ax = axes[0]
for i in range(min(3, K)):
    ax.plot(tb[mb], x[mb, i], lw=1, label="PC%d" % (i + 1))
ax.set_ylabel("PC score")
ax.set_title("population trajectory vs trials (shaded) and go cues (dashed)")
ax.legend(loc="upper right", fontsize=8)

ax = axes[1]
ax.semilogy(t_v[mv], Lv[mv], color="tab:purple", label="\u2016\u0394v\u2016 / \u0394t")
ax.semilogy(t_p[mp], Lp[mp], color="tab:green", label="\u2016\u0394p\u2016 / \u0394t")
ax.axhline(threshold, color="k", ls=":", lw=1, label="detection threshold")
sel = detected & mv
ax.plot(t_v[sel], Lv[sel], "o", color="tab:red", ms=5, label="detected jump")
ax.set_ylabel("local Lipschitz quotient")
ax.set_xlabel("time t (s)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ----------- do the brain regions move/jump together? -------------------
# Same recipe per region (correlation SVD, elbow-capped K, ||dv||/dt), then the
# traces stacked over the same zoom window. A jump present in several regions
# at the same bin is the guard-suggestive kind; a single-region jump is more
# likely local (movement, licks, one noisy population).
region_Lv, region_det = {}, {}
for reg, idx in sorted(units.groupby("region").indices.items()):
    if len(idx) < MIN_UNITS_REGION:
        continue
    ev_r, vec_r, Z_r = corr_svd(X[:, idx])
    k_r = elbow_k(ev_r, K_MAX)
    x_r = Z_r @ vec_r[:, :k_r]
    v_r = np.diff(x_r, axis=0) / dt
    Lv_r = np.linalg.norm(np.diff(v_r, axis=0), axis=1) / dt
    region_Lv[reg] = Lv_r
    region_det[reg] = t_v[Lv_r > THRESH_MULT * np.median(Lv_r)]
    print("%-28s %3d units  K=%2d  %4d jumps" % (
        reg, len(idx), k_r, len(region_det[reg])))

# how often does a global jump appear in each region (within +/- 3 bins)?
tol = 3 * dt
print("\nglobal jumps echoed per region:")
for reg, dts in region_det.items():
    if len(det_times):
        hit = np.mean([np.min(np.abs(dts - g)) <= tol if len(dts) else False
                       for g in det_times])
        print("  %-28s %5.1f%% of %d global jumps" % (reg, 100 * hit,
                                                      len(det_times)))

fig, axes = plt.subplots(len(region_Lv), 1, figsize=(13, 1.8 * len(region_Lv)),
                         sharex=True, squeeze=False)
for ax, (reg, Lv_r) in zip(axes[:, 0], region_Lv.items()):
    for _, tr in trials_df.iterrows():
        if tr["stop_time"] > z0 and tr["start_time"] < z1:
            ax.axvspan(tr["start_time"], tr["stop_time"],
                       color="tab:blue", alpha=0.07, zorder=0)
    if go_times is not None:
        for g in go_times[(go_times >= z0) & (go_times < z1)]:
            ax.axvline(g, color="0.6", ls="--", lw=0.8, zorder=0)
    for g in det_times[(det_times >= z0) & (det_times < z1)]:
        ax.axvline(g, color="tab:red", lw=0.8, alpha=0.6, zorder=1)
    ax.semilogy(t_v[mv], Lv_r[mv], color="tab:purple", lw=0.8)
    ax.set_ylabel(reg, rotation=0, ha="right", fontsize=8)
axes[-1, 0].set_xlabel("time t (s)")
axes[0, 0].set_title("per-region \u2016\u0394v\u2016/\u0394t "
                     "(red lines = global jumps, dashed = go cues)")
plt.tight_layout()
plt.show()

Caveats: raw 10 ms bins are shot-noise heavy, so the quotient baseline is set
by Poisson noise and detections are relative outliers, not true Lipschitz
blow-ups — and a jump is not yet a guard/reset (licks, movement and photostim
all move the population abruptly). Per the Hybrifolds draft, smoothing is kept
out of the pipeline on purpose: it smears resets into fake fast flows. This
notebook only asks whether massive, region-shared jumps exist at all and where
they sit relative to the trials.